# Lab 04｜旧任务中的纠正，怎样进入新任务？

先在一次支付事故分析中纠正“先发生就一定是原因”的写法，再把另一场事故交给新 Thread。最后写入 `AGENTS.md`，比较新任务收到的规则与给出的判断。

## 核心代码：把规则写入文件，让新任务按工作目录加载

> 本讲重点看下面这 10 行。它们摘自后面的代码单元格，仅供定位；运行时仍从公共准备开始按顺序执行。单元格从标题所在的第 1 格起计数，行号按格内源码计算。

**第 2 节，第 7 个单元格，第 1 至 10 行：**

```python
(WORK / "AGENTS.md").write_text("# 项目规则\n" + root_rule + "\n", encoding="utf-8")
(payment / "AGENTS.md").write_text("# 支付服务规则\n" + local_rule + "\n", encoding="utf-8")
root_start = len(lab.requests)
root_thread = lab.thread()
root_result = run(root_thread, new_prompt)
root_request = lab.request_text(root_start)
local_start = len(lab.requests)
local_thread = lab.thread(cwd=payment)
local_result = run(local_thread, new_prompt)
local_request = lab.request_text(local_start)
```

前 2 行分别写入根目录和支付服务目录的 `AGENTS.md`；`lab.thread()` 与 `lab.thread(cwd=payment)` 创建不同工作目录的会话。2 次 `request_text()` 取回实际输入，用于核对根规则与局部规则的范围。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `DEEPSEEK_API_KEY`；未设置时弹出隐藏输入框。模型通过原生 Responses 使用 DeepSeek Flash，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。准备区只启动本地连接，不调用模型。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、请求观察和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(package + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex",))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("DEEPSEEK_API_KEY", "").strip():
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DEEPSEEK_API_KEY：").strip()
if "lab" in globals() and not globals()["lab"]._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "04-agents-md")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))



WORK = lab.work
payment = WORK / "payment-service"
payment.mkdir()
old_incident = """payment-api 上次事故：01:40 连接池上限由 200 降到 60；01:50 延迟开始上升。
只有这些时间记录，尚无连接等待指标或对照数据。"""
new_incident = """payment-api 新事故：03:10 修改重试次数；03:20 错误率升高。
只有这些时间记录，尚无请求重试次数和错误来源的进一步数据。"""
correction = "时间先后不能直接证明因果。事故结论须区分已知事实、待验证的推测和还缺的依据。"
root_rule = correction  # 同一句要求，从旧任务纠正写入项目规则。
local_rule = "分析支付服务时，请把支付请求异常与下游依赖异常分开核对；尚未取得的数据明确标为未知。"
old_prompt = old_incident + "\n请为这次事故写一段简短原因分析。不要调用工具。"
new_prompt = new_incident + "\n请为这次事故写一段简短原因分析。不要调用工具。"

print("公共准备完成。")


## 1. 纠正旧任务，再交出新任务｜必读必跑

先看旧回答怎样被修正，再看全新 Thread 是否仍把先后关系写成因果。模型不一定每次犯同样的错；若这次没有再犯，如实保留结果，重点继续查新请求里有没有旧纠正。


In [ ]:
old_thread = lab.thread(cwd=payment)
old_result = run(old_thread, old_prompt)
corrected = run(old_thread, correction + "请据此修正刚才的结论。")
new_start = len(lab.requests)
unruled_thread = lab.thread(cwd=payment)
unruled_result = run(unruled_thread, new_prompt)
unruled_request = lab.request_text(new_start)
table(["阶段", "本次回答"], [
    ("旧任务初答", old_result.final_response),
    ("旧任务纠正后", corrected.final_response),
    ("新事故 / 新 Thread / 未写规则", unruled_result.final_response),
])
print("新任务输入中的旧纠正：", "存在" if correction in unruled_request else "没有自动带入")


## 2. 写入规则，再比较两个工作目录｜必读必跑

根规则约束整个项目的因果判断，支付服务子目录再补本地排查要求。使用同一份新事故、同一句任务，新建两个 Thread：根目录应只收到根规则，支付服务目录应收到两级规则。先比较输入，再读回答。


In [ ]:
(WORK / "AGENTS.md").write_text("# 项目规则\n" + root_rule + "\n", encoding="utf-8")
(payment / "AGENTS.md").write_text("# 支付服务规则\n" + local_rule + "\n", encoding="utf-8")
root_start = len(lab.requests)
root_thread = lab.thread()
root_result = run(root_thread, new_prompt)
root_request = lab.request_text(root_start)
local_start = len(lab.requests)
local_thread = lab.thread(cwd=payment)
local_result = run(local_thread, new_prompt)
local_request = lab.request_text(local_start)
table(["新 Thread 的工作目录", "根规则", "支付服务局部规则"], [
    ("项目根目录", "已装入" if root_rule in root_request else "未见", "已装入" if local_rule in root_request else "未见"),
    ("payment-service/", "已装入" if root_rule in local_request else "未见", "已装入" if local_rule in local_request else "未见"),
])
table(["同一场新事故", "原因分析的实际写法"], [
    ("未写入规则", unruled_result.final_response),
    ("只有根规则", root_result.final_response),
    ("根规则 + 支付服务规则", local_result.final_response),
])
details("扩展：两级规则在输入中的位置", json.dumps({
    "根规则位置":local_request.find(root_rule),"局部规则位置":local_request.find(local_rule),
    "旧任务纠正是否进入新任务":correction in local_request,
},ensure_ascii=False,indent=2))


## 继续观察｜扩展/可选

根规则与局部规则分别应该承担什么？新事故的回答是否仍把先后关系写成因果？规则被装入，不等于每次判断一定正确。

可把下格开关改为 `True`：撤下本次生成的局部规则，再建一个 Thread，看根规则是否仍在、局部规则是否消失。这里不推断旧 Thread 会不会即时刷新。


In [ ]:
RUN_REMOVE_LOCAL = False
if RUN_REMOVE_LOCAL:
    local_path = payment / "AGENTS.md"
    original = local_path.read_text()
    try:
        local_path.unlink()
        fresh_start = len(lab.requests)
        fresh_result = run(lab.thread(cwd=payment), new_prompt)
        fresh_request = lab.request_text(fresh_start)
        table(["撤下局部规则后的新 Thread", "实际输入"], [
            ("根规则", "仍在" if root_rule in fresh_request else "未见"),
            ("局部规则", "仍在" if local_rule in fresh_request else "未见"),
        ])
    finally:
        local_path.write_text(original,encoding="utf-8")


## 从本次观察带走什么

区分三件事：规则在哪个目录适用，是否实际进入请求，是否约束了这次判断。本例根规则与局部规则相容；不要从它推断冲突会自动解决。临时事故事实也不应升格为所有未来任务的规则。


In [ ]:
# 继续观察结束后关闭；要重跑请从公共准备开始。
lab.close()
